# MEGA-GNN Replication

This notebook tests a practical implementation of the core MEGA-GNN design for AML transaction edge classification on HI-Small. It applies two-stage multi-edge aggregation with bi-directional message passing under a 60/20/20 temporal split, then evaluates minority-class metrics. The goal is to measure how well a faithful architectural reproduction performs in this local full-graph training setup.

In [1]:
import json
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PROJECT_ROOT = Path('/content/drive/MyDrive/math-168') if IS_COLAB else Path('/Users/sophiasharif/projects/math-168')
DATA_DIR = PROJECT_ROOT / 'data'
TRANS_CSV = DATA_DIR / 'HI-Small_Trans.csv'
ACCOUNTS_CSV = DATA_DIR / 'HI-Small_accounts.csv'
if not TRANS_CSV.exists() or not ACCOUNTS_CSV.exists():
    raise FileNotFoundError(f'Missing HI-Small files in {DATA_DIR}')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.12
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
PROJECT_ROOT: /content/drive/MyDrive/math-168
DATA_DIR: /content/drive/MyDrive/math-168/data


In [2]:
# Data prep aligned to AML edge classification
def normalize_bank(s):
    return s.astype(str).str.strip().str.lstrip('0').replace('', '0')

trans = pd.read_csv(TRANS_CSV)
accounts = pd.read_csv(ACCOUNTS_CSV, usecols=['Bank ID', 'Account Number'])

src_bank = normalize_bank(trans['From Bank'])
src_acct = trans['Account'].astype(str).str.strip()
dst_bank = normalize_bank(trans['To Bank'])
dst_acct = trans['Account.1'].astype(str).str.strip()

# Paper notes four edge attributes for AML: timestamp, received amount, currency, payment format.
df_all = pd.DataFrame({
    'src': src_bank + '_' + src_acct,
    'dst': dst_bank + '_' + dst_acct,
    'timestamp': pd.to_datetime(trans['Timestamp'], errors='coerce'),
    'amount_received': pd.to_numeric(trans['Amount Received'], errors='coerce').fillna(0.0),
    'receiving_currency_code': trans['Receiving Currency'].astype('category').cat.codes,
    'payment_format_code': trans['Payment Format'].astype('category').cat.codes,
    'label': pd.to_numeric(trans['Is Laundering'], errors='coerce').fillna(0).astype(int).clip(0, 1),
}).dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

acct_bank = normalize_bank(accounts['Bank ID'])
acct_num = accounts['Account Number'].astype(str).str.strip()
acct_keys = set((acct_bank + '_' + acct_num).tolist())
print('src coverage:', round(df_all['src'].isin(acct_keys).mean(), 4))
print('dst coverage:', round(df_all['dst'].isin(acct_keys).mean(), 4))
print('full rows:', len(df_all), 'positives:', int(df_all['label'].sum()))

# Choose the smallest temporal prefix that has enough positives per split.
# This prevents extreme sparsity in train/val/test.
MIN_TOTAL_POS = 800
MIN_TRAIN_POS = 200
MIN_VAL_POS = 100
MIN_TEST_POS = 100
START_EDGES = 1_000_000
STEP_EDGES = 200_000
MAX_EDGES = 2_600_000

chosen_n = None
for n_try in range(START_EDGES, min(MAX_EDGES, len(df_all)) + 1, STEP_EDGES):
    y_try = df_all['label'].values[:n_try]
    tr_try, va_try = int(0.6 * n_try), int(0.8 * n_try)
    total_pos = int(y_try.sum())
    tr_pos = int(y_try[:tr_try].sum())
    va_pos = int(y_try[tr_try:va_try].sum())
    te_pos = int(y_try[va_try:].sum())
    if (
        total_pos >= MIN_TOTAL_POS
        and tr_pos >= MIN_TRAIN_POS
        and va_pos >= MIN_VAL_POS
        and te_pos >= MIN_TEST_POS
    ):
        chosen_n = n_try
        break

if chosen_n is None:
    chosen_n = min(MAX_EDGES, len(df_all))

# Safety cap for 16GB-class GPUs (T4) in full-graph MEGA training.
GPU_SAFE_MAX_EDGES = 1_500_000
if torch.cuda.is_available():
    chosen_n = min(chosen_n, GPU_SAFE_MAX_EDGES)

df = df_all.iloc[:chosen_n].copy().reset_index(drop=True)
print('chosen_n:', chosen_n)

nodes = pd.Index(df['src']).append(pd.Index(df['dst'])).unique().tolist()
node_to_idx = {n: i for i, n in enumerate(nodes)}
src = df['src'].map(node_to_idx).astype(np.int64).values
dst = df['dst'].map(node_to_idx).astype(np.int64).values
y = df['label'].astype(np.float32).values

# Build edge features (timestamp normalized + amount + currency + payment format)
ts = (df['timestamp'].astype('int64') / 1e9).astype(np.float64).values
ts_norm = (ts - ts.mean()) / (ts.std() + 1e-8)
edge_attr = np.stack([
    ts_norm.astype(np.float32),
    df['amount_received'].values.astype(np.float32),
    df['receiving_currency_code'].values.astype(np.float32),
    df['payment_format_code'].values.astype(np.float32),
], axis=1)
edge_attr = (edge_attr - edge_attr.mean(0, keepdims=True)) / (edge_attr.std(0, keepdims=True) + 1e-8)

# Simple initial node features (in/out degree and total amount signal)
num_nodes = len(nodes)
in_deg = np.zeros(num_nodes, dtype=np.float32)
out_deg = np.zeros(num_nodes, dtype=np.float32)
amt_sum = np.zeros(num_nodes, dtype=np.float32)
for s, d, a in zip(src, dst, df['amount_received'].values.astype(np.float32)):
    out_deg[s] += 1.0
    in_deg[d] += 1.0
    amt_sum[s] += a
node_x = np.stack([in_deg, out_deg, amt_sum], axis=1)
node_x = (node_x - node_x.mean(0, keepdims=True)) / (node_x.std(0, keepdims=True) + 1e-8)

# Build directed pair IDs for parallel-edge grouping.
pair_key = src.astype(np.int64) * np.int64(num_nodes) + dst.astype(np.int64)
unique_keys, pair_idx = np.unique(pair_key, return_inverse=True)
src_pair = (unique_keys // np.int64(num_nodes)).astype(np.int64)
dst_pair = (unique_keys % np.int64(num_nodes)).astype(np.int64)

n = len(df)
tr, va = int(0.6 * n), int(0.8 * n)
train_ids = np.arange(0, tr)
val_ids = np.arange(tr, va)
test_ids = np.arange(va, n)

print('subset rows:', n, 'positives:', int(y.sum()))
print('split positives train/val/test:', int(y[train_ids].sum()), int(y[val_ids].sum()), int(y[test_ids].sum()))
print('num_nodes:', num_nodes, 'num_pairs:', len(unique_keys))

src coverage: 1.0
dst coverage: 1.0
full rows: 5078345 positives: 5177
chosen_n: 2000000
subset rows: 2000000 positives: 984
split positives train/val/test: 361 227 396
num_nodes: 510508 num_pairs: 875696


In [3]:
# MEGA core model: two-stage aggregation + bi-directional MP + edge updates
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

src_t = torch.tensor(src, dtype=torch.long, device=device)
dst_t = torch.tensor(dst, dtype=torch.long, device=device)
pair_idx_t = torch.tensor(pair_idx, dtype=torch.long, device=device)
src_pair_t = torch.tensor(src_pair, dtype=torch.long, device=device)
dst_pair_t = torch.tensor(dst_pair, dtype=torch.long, device=device)

node_x_t = torch.tensor(node_x, dtype=torch.float32, device=device)
edge_attr_t = torch.tensor(edge_attr, dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.float32, device=device)

class MegaLayer(nn.Module):
    def __init__(self, h):
        super().__init__()
        # Eq (4): EdgeAgg over parallel edges
        self.edge_agg = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, h))
        # Eq (5)/(9): incoming and outgoing message constructors
        self.msg_in = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, h))
        self.msg_out = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, h))
        self.node_upd = nn.Sequential(nn.Linear(3 * h, h), nn.ReLU(), nn.Linear(h, h))
        # Eq (6): edge update on original edges
        self.edge_upd = nn.Sequential(nn.Linear(3 * h, h), nn.ReLU(), nn.Linear(h, h))

    def forward(self, x, e, src, dst, pair_idx, src_pair, dst_pair, num_nodes, num_pairs):
        # ---- Stage 1: parallel-edge aggregation (Eq 4) ----
        sum_e = torch.zeros((num_pairs, e.size(1)), device=e.device)
        sum_e.index_add_(0, pair_idx, e)
        cnt = torch.bincount(pair_idx, minlength=num_pairs).float().unsqueeze(1).clamp_min(1.0)
        mean_e = sum_e / cnt

        if hasattr(torch.Tensor, 'scatter_reduce'):
            idx = pair_idx.unsqueeze(1).expand(-1, e.size(1))
            max_e = torch.full((num_pairs, e.size(1)), -1e9, device=e.device)
            max_e = max_e.scatter_reduce(0, idx, e, reduce='amax', include_self=True)
            max_e = torch.where(max_e < -1e8, torch.zeros_like(max_e), max_e)
        else:
            # Fallback (rare on modern torch versions)
            max_e = mean_e

        h_pair = self.edge_agg(torch.cat([mean_e, max_e], dim=1))

        # ---- Stage 2: node-level aggregation from distinct neighbors ----
        # Incoming channel a_j (Eq 5): messages from i in Nin(j)
        m_in = self.msg_in(torch.cat([x[src_pair], h_pair], dim=1))
        a_in = torch.zeros((num_nodes, m_in.size(1)), device=e.device)
        a_in.index_add_(0, dst_pair, m_in)

        # Outgoing/reverse channel a_hat_j (Eq 9): messages from i in Nout(j)
        m_out = self.msg_out(torch.cat([x[dst_pair], h_pair], dim=1))
        a_out = torch.zeros((num_nodes, m_out.size(1)), device=e.device)
        a_out.index_add_(0, src_pair, m_out)

        x_new = self.node_upd(torch.cat([x, a_in, a_out], dim=1))

        # Edge update (Eq 6) using source node embedding and pair embedding.
        e_new = self.edge_upd(torch.cat([x[src], e, h_pair[pair_idx]], dim=1))
        return x_new, e_new

class MegaEdgeClassifier(nn.Module):
    def __init__(self, node_in=3, edge_in=4, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_proj = nn.Linear(node_in, hidden)
        self.edge_proj = nn.Linear(edge_in, hidden)
        self.layers = nn.ModuleList([MegaLayer(hidden) for _ in range(layers)])
        self.head = nn.Sequential(
            nn.Linear(hidden * 3, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, node_x, edge_attr, src, dst, pair_idx, src_pair, dst_pair, num_nodes, num_pairs):
        x = self.node_proj(node_x)
        e = self.edge_proj(edge_attr)
        for layer in self.layers:
            x, e = layer(x, e, src, dst, pair_idx, src_pair, dst_pair, num_nodes, num_pairs)
            x = F.relu(x)
            e = F.relu(e)
            x = F.dropout(x, p=self.dropout, training=self.training)
            e = F.dropout(e, p=self.dropout, training=self.training)
        return x, e

    def score_edges(self, x, e, src, dst, edge_ids):
        s = src[edge_ids]
        d = dst[edge_ids]
        ee = e[edge_ids]
        logits = self.head(torch.cat([x[s], x[d], ee], dim=1)).squeeze(1)
        return logits

    def forward(self, node_x, edge_attr, src, dst, pair_idx, src_pair, dst_pair, num_nodes, num_pairs, edge_ids):
        x, e = self.encode(node_x, edge_attr, src, dst, pair_idx, src_pair, dst_pair, num_nodes, num_pairs)
        return self.score_edges(x, e, src, dst, edge_ids)

def best_threshold(y_true, p):
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.01, 0.99, 0.01):
        f = f1_score(y_true, (p >= t).astype(int), zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

# Hyperparameters guided by paper ranges and practical memory constraints
HIDDEN = 48
LAYERS = 2
DROPOUT = 0.2
LR = 1e-3
EPOCHS = 20
PATIENCE = 4
VAL_TEST_CHUNK = 120_000
USE_AMP = torch.cuda.is_available()

model = MegaEdgeClassifier(node_in=node_x_t.size(1), edge_in=edge_attr_t.size(1), hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scaler = torch.amp.GradScaler(device='cuda', enabled=USE_AMP)

# Adaptive minority weighting: use true imbalance with a high cap for stability.
pos = float(y_t[train_ids].sum().item())
neg = float(len(train_ids) - pos)
auto_ratio = neg / max(pos, 1.0)
pos_weight = float(np.clip(auto_ratio, 1.0, 5000.0))
crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
print('train positives:', int(pos), 'negatives:', int(neg), 'auto_ratio:', round(auto_ratio, 2), 'pos_weight:', pos_weight)

def predict_probs_chunked(model, edge_ids_np, chunk_size=120_000):
    model.eval()
    out = []
    with torch.no_grad():
        x_enc, e_enc = model.encode(node_x_t, edge_attr_t, src_t, dst_t, pair_idx_t, src_pair_t, dst_pair_t, num_nodes, len(unique_keys))
        for s in range(0, len(edge_ids_np), chunk_size):
            ids = torch.tensor(edge_ids_np[s:s + chunk_size], dtype=torch.long, device=device)
            logits_c = model.score_edges(x_enc, e_enc, src_t, dst_t, ids)
            out.append(torch.sigmoid(logits_c).detach().cpu().numpy())
    return np.concatenate(out)

best_state, best_val_f1, best_thr = None, -1.0, 0.5
stale = 0
train_ids_t = torch.tensor(train_ids, dtype=torch.long, device=device)

for ep in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
        logits = model(node_x_t, edge_attr_t, src_t, dst_t, pair_idx_t, src_pair_t, dst_pair_t, num_nodes, len(unique_keys), train_ids_t)
        loss = crit(logits, y_t[train_ids_t])
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()

    pv = predict_probs_chunked(model, val_ids, chunk_size=VAL_TEST_CHUNK)
    yv = y[val_ids]
    thr, vf1 = best_threshold(yv, pv)
    print(f'epoch={ep} loss={float(loss.item()):.4f} val_f1={vf1:.4f} thr={thr:.2f}')

    if vf1 > best_val_f1:
        best_val_f1, best_thr = vf1, thr
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
    if stale >= PATIENCE:
        print('Early stopping triggered')
        break

model.load_state_dict(best_state)
p_test = predict_probs_chunked(model, test_ids, chunk_size=VAL_TEST_CHUNK)

y_test = y[test_ids]
pred = (p_test >= best_thr).astype(int)
metrics = {
    'model': 'MEGA-core',
    'rows_used': int(n),
    'num_nodes': int(num_nodes),
    'num_pairs': int(len(unique_keys)),
    'train_pos': int(y[train_ids].sum()),
    'val_pos': int(y[val_ids].sum()),
    'test_pos': int(y_test.sum()),
    'hidden': HIDDEN,
    'layers': LAYERS,
    'dropout': DROPOUT,
    'lr': LR,
    'pos_weight': pos_weight,
    'best_val_f1': float(best_val_f1),
    'best_threshold': float(best_thr),
    'test_f1': float(f1_score(y_test, pred, zero_division=0)),
    'test_precision': float(precision_score(y_test, pred, zero_division=0)),
    'test_recall': float(recall_score(y_test, pred, zero_division=0)),
    'test_pr_auc': float(average_precision_score(y_test, p_test)),
}
print(json.dumps(metrics, indent=2))

artifacts = PROJECT_ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)
with open(artifacts / 'mega_gnn_replication_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Saved:', artifacts / 'mega_gnn_replication_metrics.json')

train positives: 361 negatives: 1199639 auto_ratio: 3323.1 pos_weight: 3323.0997229916898


OutOfMemoryError: CUDA out of memory. Tried to allocate 490.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 71.75 MiB is free. Process 21124 has 104.00 MiB memory in use. Including non-PyTorch memory, this process has 14.39 GiB memory in use. Of the allocated memory 13.97 GiB is allocated by PyTorch, and 297.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)